# HGCC — editorial revision experiments (JNSPS)

Runs every experiment behind the revised Tables 7–17 and Figure 2 and writes **`/kaggle/working/results.zip`**.

**Before running**
1. *Add Input* → your Kaggle dataset containing `CODE_PACKS_updated.zip` (Kaggle unzips it; the notebook finds `processed_ccf` and `processed_cis` automatically).
2. *Settings* → **Accelerator: GPU** (T4 or P100). *Internet* can stay on (only needed if a package is missing).
3. Use **Save Version → Save & Run All (Commit)** so it keeps running in the background (stops starting new runs after 10.5 h and always writes results.zip). When it finishes, open the version → *Output* → download `results.zip`.

Do not edit the scripts: every number in the revised manuscript is generated from `results.zip`.

In [1]:
!nvidia-smi -L || echo 'NO GPU - enable Accelerator: GPU in Settings'
import importlib, subprocess, sys
for pkg, mod in [('imbalanced-learn', 'imblearn'), ('xgboost', 'xgboost')]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
import os; os.makedirs('/kaggle/working/hgcc', exist_ok=True)

GPU 0: Tesla T4 (UUID: GPU-90d28a13-d97f-a083-f51c-fe3cd808fc43)
GPU 1: Tesla T4 (UUID: GPU-7a94461f-330e-d257-97ef-9e3d70bf4a25)
torch 2.11.0+cu130 cuda True


In [2]:
%%writefile /kaggle/working/hgcc/hgcc_build_ccf.py
#!/usr/bin/env python3
"""
hgcc_build_ccf.py -- HGCC latent-group inference for the Credit Card dataset
(no identifier columns, delta = 0), with every choice fixed. This is the
implementation of Algorithm 1 used for all Credit Card results of the revised
paper.

Input : transactions with Time, V1-V28, Amount, AmountLog, Target (the columns of
        the public Kaggle file; processed_ccf/transactions.csv contains them).
Output: <out>/transactions.csv (input columns + customer_idx, card_idx,
        merchant_idx) and <out>/graph_facts.json (all settings and diagnostics).

Specification
  order      transactions sorted by Time (ties: file order); the first 80 % form
             the training period (the training split of all experiments).
  features   F = V1-V28 and AmountLog, standardised with training-period
             statistics (Time is not a clustering feature).
  co-occurrence edges
             consecutive transactions (in time order) at most 1 s apart (the
             resolution of Time), weight 1. Within the training period both
             neighbours are used; a later transaction uses only its predecessor.
  k selection (CALCULATE_OPTIMAL_CLUSTERS)
             for each entity type, mini-batch K-means (3 initialisations, batch
             4,096) is fitted on the training period for every k of a fixed grid,
             and the k with the highest silhouette coefficient on a fixed random
             sample of 20,000 training transactions is chosen.
             grids: customer {1000, 2000, 5000, 10000}; merchant {100, 250, 500,
             1000} (customers are expected to outnumber merchants, as card1
             values outnumber addr1 values 41:1 in IEEE-CIS).
  grouping   Edge-Aware K-means (edge_aware_kmeans below; identical to the
             reference implementation used for Table 5), alpha = 0.3, on the
             training period only.
  assignment of later transactions (causal)
             each validation/test transaction, in time order, takes the label c
             that minimises ||x_i - mu_c||^2 + alpha * 1[c != label of its
             predecessor] (penalty only if the predecessor is <= 1 s earlier),
             over its 10 nearest training centroids and the predecessor's label.
             Centroids are not updated, so no later transaction changes the group
             of an earlier one.
  cards      no information distinguishes a customer from its cards in this
             dataset, so card groups map one-to-one to customer groups (as in
             IEEE-CIS, where both are keyed by card1).
  connectivity  no restoration step; components are reported.
  seeds      entity type t uses seed r + {customer: 1, merchant: 2}, r = 0.
"""
import argparse
import json
import os

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse.csgraph import connected_components
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

FEATS = [f"V{i}" for i in range(1, 29)] + ["AmountLog"]
GRIDS = {"customer": [1000, 2000, 5000, 10000], "merchant": [100, 250, 500, 1000]}
SEED_OFFSET = {"customer": 1, "merchant": 2}
ALPHA, WINDOW, N_CAND, SAMPLE = 0.3, 1.0, 10, 20000


def edge_aware_kmeans(F, k, edges, alpha, seed, max_iter=20, tol=1e-4, n_cand=N_CAND):
    """J = sum ||x_i - mu_{l_i}||^2 + alpha * sum_{(i,j) in E} w_ij 1[l_i != l_j],
    minimised by alternating ICM assignment and centroid updates (same algorithm
    as edge_aware_kmeans_ref in hgcc_comments_4_6.py). Returns labels, centroids, log."""
    n = len(F)
    km = MiniBatchKMeans(k, random_state=seed, n_init=3, batch_size=4096).fit(F)
    lab, mu = km.labels_.copy(), km.cluster_centers_.copy()
    W = sparse.coo_matrix((np.ones(len(edges)), (edges[:, 0], edges[:, 1])), shape=(n, n))
    W = (W + W.T).tocsr()
    deg = np.asarray(W.sum(1)).ravel()
    rows = np.arange(n)
    prevJ, log = np.inf, []
    for it in range(max_iter):
        M = (W @ sparse.csr_matrix((np.ones(n), (rows, lab)), shape=(n, k))).tocsr()
        nb = np.asarray(M.argmax(1)).ravel()
        _, cand = NearestNeighbors(n_neighbors=min(n_cand, k)).fit(mu).kneighbors(F)
        cand = np.c_[cand, nb, lab]
        d2 = np.empty(cand.shape)
        for st in range(0, n, 20000):
            sl = slice(st, st + 20000)
            d2[sl] = ((F[sl, None, :] - mu[cand[sl]]) ** 2).sum(-1)
        same = np.asarray(M[np.repeat(rows, cand.shape[1]), cand.ravel()]).reshape(cand.shape)
        lab_new = cand[rows, (d2 + alpha * (deg[:, None] - same)).argmin(1)]
        for c in np.unique(lab_new):
            mu[c] = F[lab_new == c].mean(0)
        Wc = W.tocoo()
        cut = (lab_new[Wc.row] != lab_new[Wc.col]).sum() / 2
        J = float(((F - mu[lab_new]) ** 2).sum() + alpha * cut)
        changed = float((lab_new != lab).mean())
        log.append({"iteration": it + 1, "J": J, "share_changed": changed})
        lab = lab_new
        if changed == 0 or abs(prevJ - J) / max(J, 1e-12) < tol:
            break
        prevJ = J
    return lab, mu, log


def assign_causal(F, mu, prev_label, linked, alpha, n_cand=N_CAND):
    """Sequential assignment of later transactions (see module docstring)."""
    _, cand = NearestNeighbors(n_neighbors=min(n_cand, len(mu))).fit(mu).kneighbors(F)
    out = np.empty(len(F), dtype=np.int64)
    prev = prev_label
    for i in range(len(F)):
        c = np.append(cand[i], prev)
        cost = ((F[i] - mu[c]) ** 2).sum(1) + (alpha * (c != prev) if linked[i] else 0.0)
        out[i] = c[int(np.argmin(cost))]
        prev = out[i]
    return out


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--txn", required=True); ap.add_argument("--out", required=True)
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    t = pd.read_csv(a.txn)
    order_col = "transaction_idx" if "transaction_idx" in t else None
    t = t.sort_values(["Time"] + ([order_col] if order_col else []), kind="stable").reset_index(drop=True)
    for c in ("customer_idx", "card_idx", "merchant_idx"):
        if c in t:
            t = t.drop(columns=c)
    if "AmountLog" not in t:
        t["AmountLog"] = np.log1p(t["Amount"])
    if "Target" not in t:
        t["Target"] = t["Class"]
    n = len(t); ntr = int(0.8 * n)
    X = t[FEATS].values.astype(np.float64)
    F = (X - X[:ntr].mean(0)) / (X[:ntr].std(0) + 1e-6)
    gap = np.r_[np.inf, np.diff(t["Time"].values)]
    linked = gap <= WINDOW                                   # linked to its predecessor
    i = np.where(linked[1:ntr])[0] + 1
    E = np.c_[i - 1, i]                                      # training-period co-occurrence edges
    rng = np.random.default_rng(a.seed)
    sample = rng.choice(ntr, SAMPLE, replace=False)
    facts = {"n_transactions": n, "n_training": ntr, "alpha": ALPHA, "window_s": WINDOW,
             "training_cooccurrence_edges": int(len(E)), "share_linked_to_predecessor": float(linked[1:].mean()),
             "grids": GRIDS, "silhouette": {}, "chosen_k": {}, "eakm_log": {}}
    labels = {}
    for typ in ("customer", "merchant"):
        seed = a.seed + SEED_OFFSET[typ]
        sil = {}
        for k in GRIDS[typ]:
            km = MiniBatchKMeans(k, random_state=seed, n_init=3, batch_size=4096).fit(F[:ntr])
            sil[k] = float(silhouette_score(F[:ntr][sample], km.labels_[sample]))
            print(f"{typ} k={k}: silhouette {sil[k]:.4f}", flush=True)
        k = max(sil, key=sil.get)
        lab_tr, mu, log = edge_aware_kmeans(F[:ntr], k, E, ALPHA, seed)
        lab_late = assign_causal(F[ntr:], mu, lab_tr[-1], linked[ntr:], ALPHA)
        lab = np.r_[lab_tr, lab_late]
        _, lab = np.unique(lab, return_inverse=True)        # contiguous indices
        labels[typ] = lab
        facts["silhouette"][typ] = sil; facts["chosen_k"][typ] = k; facts["eakm_log"][typ] = log
        print(f"{typ}: chosen k = {k}, groups used = {lab.max() + 1}, EAKM iterations = {len(log)}", flush=True)
    t["customer_idx"] = labels["customer"]; t["card_idx"] = labels["customer"]; t["merchant_idx"] = labels["merchant"]
    t.to_csv(os.path.join(a.out, "transactions.csv"), index=False)

    # diagnostics (same definitions as hgcc_ccf_control.py diagnose)
    nt, nc, nm = n, int(t.customer_idx.max() + 1), int(t.merchant_idx.max() + 1)
    ti = np.arange(nt)
    cu, ca, me = nt, nt + nc, nt + 2 * nc                    # node offsets: transactions, customers, cards, merchants
    src = np.r_[t.customer_idx.values + cu, t.card_idx.values + ca, ti, np.arange(nc) + cu]       # Makes, Used_In, At_Merchant, Owns
    dst = np.r_[ti, ti, t.merchant_idx.values + me, np.arange(nc) + ca]
    N = nt + 2 * nc + nm
    ncomp, comp = connected_components(sparse.coo_matrix((np.ones(len(src)), (src, dst)), shape=(N, N)), directed=False)
    facts["graph"] = {"transactions": nt, "customers": nc, "cards": nc, "merchants": nm, "nodes": N,
                      "owns": nc, "makes": nt, "used_in": nt, "at_merchant": nt, "edges": 3 * nt + nc,
                      "components": int(ncomp), "largest_component": int(np.bincount(comp).max())}
    p = float(t.Target.mean())
    for e in ("customer_idx", "merchant_idx"):
        g = t.groupby(e)
        fr = g["Target"].agg(["sum", "size"])
        facts[e] = {"n_groups": int(t[e].nunique()), "median_size": float(g.size().median()),
                    "min_size": int(g.size().min()), "max_size": int(g.size().max()),
                    "consecutive_same_group": float((t[e].values[1:] == t[e].values[:-1]).mean()),
                    "median_time_span_s": float((g["Time"].max() - g["Time"].min()).median()),
                    "groups_with_fraud": int((fr["sum"] > 0).sum()),
                    "expected_groups_with_fraud_if_random": float((1 - (1 - p) ** fr["size"]).sum())}
        ff = (fr["sum"] * (fr["sum"] - 1)).sum(); f_ = (fr["sum"] * (fr["size"] - 1)).sum()
        facts[e]["P_other_fraud_given_fraud"] = float(ff / f_) if f_ else float("nan")
    facts["base_fraud_rate"] = p
    json.dump(facts, open(os.path.join(a.out, "graph_facts.json"), "w"), indent=1)
    print(json.dumps({k: v for k, v in facts.items() if k not in ("eakm_log",)}, indent=1))


if __name__ == "__main__":
    main()


Writing /kaggle/working/hgcc/hgcc_build_ccf.py


In [3]:
%%writefile /kaggle/working/hgcc/hgcc_editorial_experiments.py
#!/usr/bin/env python3
"""
hgcc_editorial_experiments.py -- single, leakage-controlled evaluation of the
HGCC graphs, written in response to the editorial comments (JNSPS).

Every number in the revised result tables is produced by this script (runs)
and by `hgcc_editorial_tables.py` (aggregation, tests, confusion matrices).
Nothing is typed by hand.

Data (both are HGCC outputs released in CODE PACKS):
  * Credit Card : graph built by hgcc_build_ccf.py (--ccf-file), i.e. the public
                  columns plus customer_idx, card_idx and merchant_idx
  * IEEE-CIS    : processed_cis/{transactions, edges_makes, edges_used_in,
                  edges_at_merchant}.csv

Protocol (identical for every model and graph variant):
  * all transactions are put in arrival order (Time / TransactionDT, ties by
    row order); "earlier" below means earlier in this order.
  * split  temporal : first 80 % train, next 10 % validation, last 10 % test.
           random   : stratified random 80/10/10, redrawn for every seed (used
                      only to reconcile the original random-split ablation).
  * transaction features are standardised with training-split statistics;
  * entity-node features (log count, log mean amount, log total amount) are
    computed from TRAINING transactions only;
  * CAUSAL message passing: when transaction i is represented, each of its
    entity nodes (customer, card, merchant) aggregates only transactions that
    arrived no later than i (exact cumulative mean / cumulative attention
    within each entity, in arrival order). A prediction therefore uses only
    the transaction itself, earlier transactions and training-period entity
    statistics -- never a later transaction and never any label;
  * labels: training labels -> loss; validation labels -> early stopping and
    decision threshold; test labels -> reported metrics only;
  * decision threshold = value maximising F1 on the validation split, applied
    unchanged to the test split;
  * graph variants: G0 = HGCC graph; G1 = the (customer, card, merchant)
    tuple of every transaction randomly permuted across transactions (entity
    sizes and node degrees preserved, relational information destroyed; one
    permutation per seed); G2 = no entity nodes (transaction features only).

Models (heterogeneous transaction-entity graph; Owns edges are not used for
message passing because each transaction's customer-card link is already
carried by its Makes and Used_In edges):
  sage : 2-layer GraphSAGE (mean aggregation, separate weights per relation),
         hidden 32, dropout 0.2.
  gat  : 2-layer GAT, 4 heads x 8, attention of an entity over its earlier
         transactions, skip connection, LayerNorm, ELU, hidden 32,
         dropout 0.3, MLP head.
  grnn : T = 3 recurrent message-passing steps with a GRUCell state update
         per node type (Eqs. 24-27), hidden 32, dropout 0.2, MLP head.
  rf / xgb / knn : tabular baselines on the same transaction features,
         SMOTE applied to the training split only.
  GNNs: focal loss (alpha 0.25, gamma 2), Adam (learning rate per model chosen on
  validation, see run_kaggle.py; weight decay 1e-5),
  gradient-norm clipping at 1.0, full batch, <= 500 epochs, validation PR-AUC every 5 epochs, early stopping after
  20 checks without improvement but not before epoch 100; best-validation weights restored.

Usage
  python hgcc_editorial_experiments.py --dataset ccf --data "CODE PACKS" \
      --split temporal --models sage gat grnn --variants G0 G1 G2 \
      --seeds 0 1 2 3 4 5 6 7 8 9 --out runs/ccf_temporal.csv
"""
import argparse
import os
import time

import numpy as np
import pandas as pd

LR = {"sage": 0.003, "gat": 0.01, "grnn": 0.003}  # chosen from {0.003, 0.01} by validation PR-AUC, Credit Card seed 0
REL = ("customer", "card", "merchant")
DEV = "cpu"   # set in main(): "cuda" when available


# ----------------------------------------------------------------------------- data
def load_dataset(name, root, ccf_file=None):
    """One row per transaction, in arrival order: features, label, the HGCC
    entity index of the transaction (customer, card, merchant) and amount."""
    t0 = time.perf_counter()
    if name == "ccf":
        t = pd.read_csv(ccf_file or os.path.join(root, "processed_ccf", "transactions.csv"))
        t = t.sort_values(["Time", "transaction_idx"], kind="stable").reset_index(drop=True)
        feats = [f"V{i}" for i in range(1, 29)] + ["AmountLog"]
        X = t[feats].astype(np.float32).values
        ent = {r: t[f"{r}_idx"].values.astype(np.int64) for r in REL}
        y, amt = t["Target"].values.astype(np.int64), t["Amount"].values.astype(np.float64)
    else:
        d = os.path.join(root, "processed_cis")
        t = pd.read_csv(os.path.join(d, "transactions.csv"))
        t = t.iloc[np.lexsort((np.arange(len(t)), t["TransactionDT"].values))].reset_index(drop=True)
        src = {"customer": ("edges_makes.csv", "Target", "Source"),
               "card": ("edges_used_in.csv", "Target", "Source"),
               "merchant": ("edges_at_merchant.csv", "Source", "Target")}
        ent = {}
        for r, (f, txn_col, ent_col) in src.items():
            e = pd.read_csv(os.path.join(d, f))
            s = pd.Series(e[ent_col].values, index=e[txn_col].values)
            assert s.index.is_unique, f"a transaction has more than one {r}"
            codes, _ = pd.factorize(s.reindex(t["TransactionID"].values).values)
            assert (codes >= 0).all(), f"transaction without {r}"
            ent[r] = codes.astype(np.int64)
        tod = (t["TransactionDT"].values % 86400) / 86400.0 * 2 * np.pi
        prod = pd.get_dummies(t["ProductCD"].fillna("NA"), prefix="P").astype(np.float32)
        X = np.column_stack([t["AmountLog"].values, np.sin(tod), np.cos(tod), prod.values]).astype(np.float32)
        y, amt = t["Target"].values.astype(np.int64), t["Amount"].values.astype(np.float64)
    return {"X": X, "y": y, "ent": ent, "amt": amt, "load_seconds": time.perf_counter() - t0}


def make_split(D, kind, seed):
    n = len(D["y"])
    split = np.empty(n, dtype=object)
    if kind == "temporal":                      # rows are already in arrival order
        a, b = int(0.8 * n), int(0.9 * n)
        split[:a], split[a:b], split[b:] = "train", "val", "test"
    else:
        from sklearn.model_selection import train_test_split
        idx = np.arange(n)
        tr, rest = train_test_split(idx, test_size=0.2, random_state=seed, stratify=D["y"])
        va, te = train_test_split(rest, test_size=0.5, random_state=seed, stratify=D["y"][rest])
        split[tr], split[va], split[te] = "train", "val", "test"
    return {k: split == k for k in ("train", "val", "test")}


def std_train(X, tr):
    mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-6
    return ((X - mu) / sd).astype(np.float32)


def entity_feats(ids, amt, tr):
    """log count, log mean amount, log total amount over TRAINING transactions only."""
    n = ids.max() + 1
    df = pd.DataFrame({"id": ids[tr], "a": amt[tr]}).groupby("id")["a"].agg(["size", "mean", "sum"])
    df = df.reindex(range(n)).fillna(0.0)
    F = np.column_stack([np.log1p(df["size"].values), np.log1p(df["mean"].values), np.log1p(df["sum"].values)])
    return ((F - F.mean(0)) / (F.std(0) + 1e-6)).astype(np.float32)


class CausalGroups:
    """For one relation: transactions grouped by entity, arrival order kept.
    cumsum(H)[i] = sum of H[j] over transactions j of i's entity with j <= i
                   (custom autograd: the gradient is the reverse within-group
                   cumulative sum, so no intermediate tensors are stored);
    mean(H)[i]   = cumsum(H)[i] / (number of such j);
    attend(S,V)  = softmax(S)-weighted sum of V with the same causal restriction.
    Sums are accumulated in float64 and returned in float32."""

    def __init__(self, ent):
        import torch
        n = len(ent)
        perm = np.argsort(ent, kind="stable")              # grouped by entity, arrival order inside
        es = ent[perm]
        start = np.r_[True, es[1:] != es[:-1]]
        end = np.r_[es[1:] != es[:-1], True]
        base = np.maximum.accumulate(np.where(start, np.arange(n), 0))
        last = np.minimum.accumulate(np.where(end, np.arange(n), n)[::-1])[::-1]
        self.perm = torch.from_numpy(perm).to(DEV)
        self.inv = torch.from_numpy(np.argsort(perm)).to(DEV)
        self.base = torch.from_numpy(base).to(DEV)                  # sorted position of the group's first row
        self.last = torch.from_numpy(last.copy()).to(DEV)                 # sorted position of the group's last row
        cnt_sorted = (np.arange(n) - base + 1).astype(np.float32)
        self.cnt = torch.from_numpy(cnt_sorted[np.argsort(perm)]).unsqueeze(1).to(DEV)   # arrival order
        self.ent = torch.from_numpy(ent).to(DEV)
        self._fn = self._make_fn()

    def _scan(self, H, reverse=False):
        """inclusive within-group cumulative sum (reverse: from the group's end)."""
        import torch
        flat = H.reshape(H.size(0), -1)
        Hs = flat.index_select(0, self.perm).double()
        cs = torch.cumsum(Hs, 0)
        pad = torch.cat([torch.zeros_like(cs[:1]), cs], 0)
        fwd = cs - pad.index_select(0, self.base)
        if reverse:   # sum_{k >= i in group} = group total - forward sum + own value
            out = (cs.index_select(0, self.last) - pad.index_select(0, self.base)) - fwd + Hs
        else:
            out = fwd
        return out.to(H.dtype).index_select(0, self.inv).reshape(H.shape)

    def _make_fn(self):
        import torch
        grp = self
        class GroupCumsum(torch.autograd.Function):
            @staticmethod
            def forward(ctx, H):
                return grp._scan(H)
            @staticmethod
            def backward(ctx, G):
                return grp._scan(G.contiguous(), reverse=True)
        return GroupCumsum.apply

    def cumsum(self, H):
        return self._fn(H)

    def mean(self, H):
        return self._fn(H) / self.cnt

    def attend(self, S, V):
        """S: N x heads scores, V: N x heads x d values (arrival order)."""
        import torch
        # logits soft-capped at +-30, so every weight depends only on its own
        # transaction (no per-entity shift, which would involve later rows)
        w = torch.exp(30.0 * torch.tanh(S / 30.0))
        num = self._fn((w.unsqueeze(-1) * V).contiguous())
        den = self._fn(w.contiguous()).unsqueeze(-1)
        return num / den


def build_inputs(D, masks, variant, seed):
    import torch
    t0 = time.perf_counter()
    tr = masks["train"]
    G = {"x_txn": torch.from_numpy(std_train(D["X"], tr)).to(DEV), "rel": {}}
    if variant != "G2":
        ent = dict(D["ent"])
        if variant == "G1":
            p = np.random.default_rng(1000 + seed).permutation(len(D["y"]))
            ent = {r: ent[r][p] for r in REL}
        for r in REL:
            G["rel"][r] = {"groups": CausalGroups(ent[r]), "idx": torch.from_numpy(ent[r]).to(DEV),
                           "x": torch.from_numpy(entity_feats(ent[r], D["amt"], tr)).to(DEV)}
    return G, time.perf_counter() - t0


# ----------------------------------------------------------------------------- models
def make_model(name, G):
    import torch
    import torch.nn.functional as F
    nn = torch.nn
    rels = list(G["rel"])
    dx = G["x_txn"].size(1)
    dent = {r: G["rel"][r]["x"].size(1) for r in rels}

    class SAGE(nn.Module):
        """h_e^1(i) = relu(W_root h_e^0 + W_nb mean_{j<=i, j in e} h_j^0)
           h_i^l    = relu(W_root^l h_i^{l-1} + sum_r W_r^l h_{e_r(i)}^{l-1})"""
        def __init__(s, h=32, dp=0.2):
            super().__init__()
            s.inp = nn.Linear(dx, h)
            s.einp = nn.ModuleDict({r: nn.Linear(dent[r], h) for r in rels})
            s.e_root = nn.ModuleDict({r: nn.Linear(h, h) for r in rels})
            s.e_nb = nn.ModuleDict({r: nn.Linear(h, h, bias=False) for r in rels})
            s.t_root = nn.ModuleList([nn.Linear(h, h) for _ in range(2)])
            s.t_nb = nn.ModuleList([nn.ModuleDict({r: nn.Linear(h, h, bias=False) for r in rels}) for _ in range(2)])
            s.head, s.dp = nn.Linear(h, 1), dp
        def forward(s, G):
            D = lambda z: F.dropout(z, s.dp, s.training)
            t0 = D(F.relu(s.inp(G["x_txn"])))
            e0 = {r: D(F.relu(s.einp[r](G["rel"][r]["x"])))[G["rel"][r]["idx"]] for r in rels}
            e1 = {r: D(F.relu(s.e_root[r](e0[r]) + s.e_nb[r](G["rel"][r]["groups"].mean(t0)))) for r in rels}
            t1 = D(F.relu(s.t_root[0](t0) + sum((s.t_nb[0][r](e0[r]) for r in rels), torch.zeros_like(t0))))
            t2 = D(F.relu(s.t_root[1](t1) + sum((s.t_nb[1][r](e1[r]) for r in rels), torch.zeros_like(t1))))
            return s.head(t2).squeeze(-1)

    class GAT(nn.Module):
        """Entity e attends over its transactions j <= i (GAT scores, 4 heads);
           a transaction receives one message per relation plus a skip term."""
        def __init__(s, h=32, heads=4, dp=0.3):
            super().__init__()
            s.h, s.H, s.dh, s.dp = h, heads, h // heads, dp
            s.inp = nn.Linear(dx, h)
            s.einp = nn.ModuleDict({r: nn.Linear(dent[r], h) for r in rels})
            s.Ws = nn.ModuleDict({r: nn.Linear(h, h, bias=False) for r in rels})      # source (transaction) transform
            s.Wd = nn.ModuleDict({r: nn.Linear(h, h, bias=False) for r in rels})      # destination (entity) transform
            s.a_s = nn.ParameterDict({r: nn.Parameter(torch.randn(heads, h // heads) * 0.1) for r in rels})
            s.a_d = nn.ParameterDict({r: nn.Parameter(torch.randn(heads, h // heads) * 0.1) for r in rels})
            s.e_root = nn.ModuleDict({r: nn.Linear(h, h) for r in rels})
            s.t_root = nn.ModuleList([nn.Linear(h, h) for _ in range(2)])
            s.t_msg = nn.ModuleList([nn.ModuleDict({r: nn.Linear(h, h, bias=False) for r in rels}) for _ in range(2)])
            s.norm = nn.ModuleList([nn.LayerNorm(h) for _ in range(2)])
            s.enorm = nn.ModuleDict({r: nn.LayerNorm(h) for r in rels})
            s.head = nn.Sequential(nn.Linear(h, h // 2), nn.ReLU(), nn.Dropout(dp), nn.Linear(h // 2, 1))
        def forward(s, G):
            D = lambda z: F.dropout(z, s.dp, s.training)
            t0 = D(F.relu(s.inp(G["x_txn"])))
            e0 = {r: D(F.relu(s.einp[r](G["rel"][r]["x"])))[G["rel"][r]["idx"]] for r in rels}
            e1 = {}
            for r in rels:
                V = s.Ws[r](t0).view(-1, s.H, s.dh)
                Q = s.Wd[r](e0[r]).view(-1, s.H, s.dh)
                # score of source j for its entity: a_s.W_s h_j + a_d.W_d h_e (entity query = static e0)
                Sj = (V * s.a_s[r]).sum(-1) + (Q * s.a_d[r]).sum(-1)
                agg = G["rel"][r]["groups"].attend(F.leaky_relu(Sj, 0.2), V).flatten(1)
                e1[r] = D(F.elu(s.enorm[r](s.e_root[r](e0[r]) + agg)))
            t1 = D(F.elu(s.norm[0](s.t_root[0](t0) + sum((s.t_msg[0][r](e0[r]) for r in rels), torch.zeros_like(t0)))))
            t2 = D(F.elu(s.norm[1](s.t_root[1](t1) + sum((s.t_msg[1][r](e1[r]) for r in rels), torch.zeros_like(t1)))))
            return s.head(t2).squeeze(-1)

    class GRNN(nn.Module):
        """T recurrent steps: e_r^k(i) = GRU(W mean_{j<=i, j in e} t_j^{k-1}, e_r^{k-1}(i));
           t_i^k = GRU(sum_r W_r e_r^{k-1}(i), t_i^{k-1})."""
        def __init__(s, h=32, steps=3, dp=0.2):
            super().__init__()
            s.inp = nn.Linear(dx, h)
            s.einp = nn.ModuleDict({r: nn.Linear(dent[r], h) for r in rels})
            s.e_msg = nn.ModuleDict({r: nn.Linear(h, h) for r in rels})
            s.t_msg = nn.ModuleDict({r: nn.Linear(h, h, bias=False) for r in rels})
            s.t_self = nn.Linear(h, h)
            s.e_gru = nn.ModuleDict({r: nn.GRUCell(h, h) for r in rels})
            s.t_gru = nn.GRUCell(h, h)
            s.steps, s.dp = steps, dp
            s.head = nn.Sequential(nn.Linear(h, h // 2), nn.ReLU(), nn.Dropout(dp), nn.Linear(h // 2, 1))
        def forward(s, G):
            D = lambda z: F.dropout(z, s.dp, s.training)
            from torch.utils.checkpoint import checkpoint
            t = F.relu(s.inp(G["x_txn"]))
            e = [F.relu(s.einp[r](G["rel"][r]["x"]))[G["rel"][r]["idx"]] for r in rels]
            def step(t, *e):
                m_t = s.t_self(t) + sum((s.t_msg[r](e[k]) for k, r in enumerate(rels)), torch.zeros_like(t))
                e = [D(s.e_gru[r](F.relu(s.e_msg[r](G["rel"][r]["groups"].mean(t))), e[k])) for k, r in enumerate(rels)]
                return (D(s.t_gru(F.relu(m_t), t)), *e)
            for _ in range(s.steps):   # activation checkpointing: per-transaction entity states are memory-heavy
                out = checkpoint(step, t, *e, use_reentrant=False) if s.training else step(t, *e)
                t, e = out[0], list(out[1:])
            return s.head(t).squeeze(-1)

    return {"sage": SAGE, "gat": GAT, "grnn": GRNN}[name]()


def focal(logit, y, a=0.25, gm=2.0):
    import torch
    import torch.nn.functional as F
    p = torch.sigmoid(logit)
    ce = F.binary_cross_entropy_with_logits(logit, y, reduction="none")
    pt = p * y + (1 - p) * (1 - y)
    at = a * y + (1 - a) * (1 - y)
    return (at * (1 - pt) ** gm * ce).mean()


def train_gnn(name, G, y, masks, seed, lr, epochs=500, patience=20, every=5, min_epochs=100, ckpt=None):
    import torch
    from sklearn.metrics import average_precision_score
    torch.manual_seed(seed); np.random.seed(seed)
    model = make_model(name, G).to(DEV)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    yt = torch.from_numpy(y.astype(np.float32)).to(DEV)
    mtr = torch.from_numpy(masks["train"]).to(DEV)
    yv = y[masks["val"]]
    best, best_state, bad, ep = -1.0, None, 0, 0
    t0 = time.perf_counter(); eval_s = 0.0
    for ep in range(epochs):
        model.train(); opt.zero_grad()
        loss = focal(model(G)[mtr], yt[mtr])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        if (ep + 1) % every and ep != epochs - 1:
            continue
        te = time.perf_counter()
        model.eval()
        with torch.no_grad():
            pv = torch.sigmoid(model(G)).cpu().numpy()[masks["val"]]
        ap = average_precision_score(yv, pv)
        eval_s += time.perf_counter() - te
        if ap > best:
            best, bad = ap, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience and ep + 1 >= min_epochs:
                break
    train_s = time.perf_counter() - t0
    model.load_state_dict(best_state); model.eval()
    t1 = time.perf_counter()
    with torch.no_grad():
        p = torch.sigmoid(model(G)).cpu().numpy()
    infer_s = time.perf_counter() - t1
    if not np.isfinite(p).all():
        raise FloatingPointError(f"non-finite predictions ({name}, seed {seed})")
    if ckpt:
        torch.save({"state_dict": best_state, "model": name, "seed": seed, "lr": lr}, ckpt)
    return p, {"epochs_run": ep + 1, "best_val_pr_auc": best, "train_seconds": train_s,
               "val_eval_seconds_within_training": eval_s, "inference_seconds": infer_s,
               "n_params": sum(q.numel() for q in model.parameters())}


def run_baseline(name, D, masks, seed):
    from imblearn.over_sampling import SMOTE
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.neighbors import KNeighborsClassifier
    from xgboost import XGBClassifier
    tr = masks["train"]
    X = std_train(D["X"], tr)
    t0 = time.perf_counter()
    Xs, ys = SMOTE(random_state=seed).fit_resample(X[tr], D["y"][tr])
    clf = {"rf": lambda: RandomForestClassifier(n_estimators=200, min_samples_leaf=2, n_jobs=2, random_state=seed),
           "xgb": lambda: XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
                                        colsample_bytree=0.8, tree_method="hist", n_jobs=2, random_state=seed),
           "knn": lambda: KNeighborsClassifier(n_neighbors=5, n_jobs=2)}[name]()
    clf.fit(Xs, ys)
    train_s = time.perf_counter() - t0
    t1 = time.perf_counter()
    if name == "knn":   # score only the rows that are used (val, test)
        p = np.full(len(X), np.nan)
        ev = masks["val"] | masks["test"]
        p[ev] = clf.predict_proba(X[ev])[:, 1]
    else:
        p = clf.predict_proba(X)[:, 1]
    return p, {"epochs_run": np.nan, "best_val_pr_auc": np.nan, "train_seconds": train_s,
               "val_eval_seconds_within_training": 0.0, "inference_seconds": time.perf_counter() - t1,
               "n_params": np.nan}


# ----------------------------------------------------------------------------- metrics
def evaluate(p, y, masks):
    from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
    yv, pv = y[masks["val"]], p[masks["val"]]
    pr, rc, th = precision_recall_curve(yv, pv)
    f1s = 2 * pr[:-1] * rc[:-1] / np.clip(pr[:-1] + rc[:-1], 1e-12, None)
    thr = float(th[np.argmax(f1s)])
    yt, pt = y[masks["test"]], p[masks["test"]]
    pred = pt >= thr
    tp = int((pred & (yt == 1)).sum()); fp = int((pred & (yt == 0)).sum())
    fn = int((~pred & (yt == 1)).sum()); tn = int((~pred & (yt == 0)).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"threshold": thr, "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "accuracy": (tp + tn) / len(yt), "precision": prec, "recall": rec, "f1": f1,
            "roc_auc": float(roc_auc_score(yt, pt)), "pr_auc": float(average_precision_score(yt, pt)),
            "n_test": int(len(yt)), "n_test_fraud": int(yt.sum())}


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--dataset", choices=["ccf", "ieee"], required=True)
    ap.add_argument("--data", required=True, help="CODE PACKS directory")
    ap.add_argument("--split", choices=["temporal", "random"], default="temporal")
    ap.add_argument("--models", nargs="+", default=["sage", "gat", "grnn"])
    ap.add_argument("--variants", nargs="+", default=["G0", "G1", "G2"])
    ap.add_argument("--seeds", type=int, nargs="+", default=list(range(10)))
    ap.add_argument("--epochs", type=int, default=500)
    ap.add_argument("--lr", type=float, default=None, help="learning rate (all GNNs in this call)")
    ap.add_argument("--threads", type=int, default=2)
    ap.add_argument("--ccf-file", default=None, help="Credit Card graph built by hgcc_build_ccf.py (transactions.csv)")
    ap.add_argument("--device", default="auto", help="auto | cpu | cuda")
    ap.add_argument("--out", required=True)
    ap.add_argument("--pred-dir", default=None, help="save test/val probabilities per run (.npz)")
    ap.add_argument("--ckpt-dir", default=None, help="save model checkpoints per run (.pt)")
    a = ap.parse_args()
    global DEV
    if set(a.models) & {"sage", "gat", "grnn"}:
        import torch
        torch.set_num_threads(a.threads)
        DEV = ("cuda" if torch.cuda.is_available() else "cpu") if a.device == "auto" else a.device
    for d in (a.pred_dir, a.ckpt_dir, os.path.dirname(a.out)):
        if d:
            os.makedirs(d, exist_ok=True)
    D = load_dataset(a.dataset, a.data, a.ccf_file)
    done = pd.read_csv(a.out) if os.path.exists(a.out) else pd.DataFrame(columns=["model", "variant", "seed"])
    for s in a.seeds:
        masks = make_split(D, a.split, s)
        for m in a.models:
            for v in (a.variants if m in ("sage", "gat", "grnn") else ["tabular"]):
                key = (done["model"] == m) & (done["variant"] == v) & (done["seed"] == s)
                if key.any():
                    continue
                tag = f"{a.dataset}_{a.split}_{m}_{v}_s{s}"
                t_all = time.perf_counter()
                if m in ("sage", "gat", "grnn"):
                    G, build_s = build_inputs(D, masks, v, s)
                    lr = a.lr if a.lr else LR[m]
                    p, info = train_gnn(m, G, D["y"], masks, s, lr, epochs=a.epochs,
                                        ckpt=os.path.join(a.ckpt_dir, tag + ".pt") if a.ckpt_dir else None)
                else:
                    build_s, lr = 0.0, np.nan
                    p, info = run_baseline(m, D, masks, s)
                r = evaluate(p, D["y"], masks)
                total = time.perf_counter() - t_all
                r.update(info)
                r.update({"dataset": a.dataset, "split": a.split, "model": m, "variant": v, "seed": s, "lr": lr,
                          "load_seconds_once": D["load_seconds"], "graph_build_seconds": build_s,
                          "run_seconds_total": total,
                          "other_seconds": total - build_s - info["train_seconds"] - info["inference_seconds"]})
                if a.pred_dir:
                    np.savez_compressed(os.path.join(a.pred_dir, tag + ".npz"),
                                        p_test=p[masks["test"]].astype(np.float32), y_test=D["y"][masks["test"]].astype(np.int8),
                                        p_val=p[masks["val"]].astype(np.float32), y_val=D["y"][masks["val"]].astype(np.int8),
                                        threshold=r["threshold"])
                print({k: (round(x, 4) if isinstance(x, float) else x) for k, x in r.items()}, flush=True)
                done = pd.concat([done, pd.DataFrame([r])], ignore_index=True)
                done.to_csv(a.out, index=False)


if __name__ == "__main__":
    main()


Writing /kaggle/working/hgcc/hgcc_editorial_experiments.py


In [4]:
%%writefile /kaggle/working/hgcc/hgcc_editorial_tables.py
#!/usr/bin/env python3
"""
hgcc_editorial_tables.py -- builds every revised table from the per-run CSVs
written by hgcc_editorial_experiments.py and checks their internal
consistency (F1 vs precision/recall, confusion matrix vs metrics, stage times
vs end-to-end time).

Conventions (stated in every table caption of the manuscript):
  * "mean (SD)" = metric computed separately for each run (seed) on that
    run's test split, then averaged over runs. Because F1 is a non-linear
    function of precision and recall, the mean F1 is NOT the F1 of the mean
    precision and mean recall.
  * confusion matrices are the counts of ONE run (seed 0) on its held-out
    test split; the metrics printed next to them are computed from those
    counts (aggregated predictions of that run).
  * paired tests: exact two-sided Wilcoxon signed-rank test over seeds
    (scipy, method="exact"), Holm correction within each family.

Usage: python hgcc_editorial_tables.py --runs runs/ --out tables/
"""
import argparse
import glob
import os

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

METRICS = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
NAMES = {"sage": "GraphSAGE", "gat": "GAT", "grnn": "GRNN", "rf": "Random Forest", "xgb": "XGBoost", "knn": "K-NN"}
DS = {"ccf": "Credit Card", "ieee": "IEEE-CIS"}


def holm(p):
    p = np.asarray(p, float); m = len(p); o = np.argsort(p)
    adj = np.empty(m); run = 0.0
    for i, j in enumerate(o):
        run = max(run, min(1.0, (m - i) * p[j])); adj[j] = run
    return adj


def ms(x, d=4):
    x = np.asarray(x, float)
    return f"{x.mean():.{d}f} ({x.std(ddof=1):.{d}f})" if len(x) > 1 else f"{x.mean():.{d}f}"


def check_runs(r):
    """Every run: F1 equals 2PR/(P+R); metrics equal those implied by tp/fp/fn/tn."""
    P = r.tp / (r.tp + r.fp).replace(0, np.nan); R = r.tp / (r.tp + r.fn)
    F = 2 * P * R / (P + R)
    A = (r.tp + r.tn) / (r.tp + r.fp + r.fn + r.tn)
    for name, a, b in [("precision", r.precision, P.fillna(0)), ("recall", r.recall, R), ("f1", r.f1, F.fillna(0)),
                       ("accuracy", r.accuracy, A), ("f1_from_PR", r.f1,
                        (2 * r.precision * r.recall / (r.precision + r.recall)).fillna(0))]:
        bad = (a - b).abs() > 1e-9
        assert not bad.any(), f"inconsistent {name} in {bad.sum()} runs"
    assert ((r.tp + r.fp + r.fn + r.tn) == r.n_test).all()
    assert ((r.tp + r.fn) == r.n_test_fraud).all()


def perf_table(r, rows, title):
    out = []
    for (ds, model, variant) in rows:
        x = r[(r.dataset == ds) & (r.model == model) & (r.variant == variant) & (r.split == "temporal")]
        if x.empty:
            continue
        rec = {"Dataset": DS[ds], "Model": NAMES[model], "Seeds": len(x)}
        rec.update({m: ms(x[m]) for m in METRICS})
        out.append(rec)
    t = pd.DataFrame(out); t.attrs["title"] = title
    return t


def paired(r, ds, split, model, ref, others, metrics=("pr_auc", "f1")):
    rows = []
    a = r[(r.dataset == ds) & (r.split == split) & (r.model == model)]
    for v in others:
        for m in metrics:
            x = a[a.variant == ref].set_index("seed")[m]; y = a[a.variant == v].set_index("seed")[m]
            s = x.index.intersection(y.index); d = (x[s] - y[s]).values
            if len(s) < 2:
                continue
            if len(s) < 6:   # exact two-sided Wilcoxon cannot reach p < 0.05 with n < 6
                rows.append({"Dataset": DS[ds], "Split": split, "Model": NAMES[model], "Comparison": f"{ref} vs {v}",
                             "Metric": "PR-AUC" if m == "pr_auc" else "F1", "n": len(s), "Mean diff": d.mean(),
                             "Median diff": np.median(d), "W": np.nan, "Exact p": np.nan})
                continue
            st = wilcoxon(x[s].values, y[s].values, method="exact") if np.any(d != 0) else None
            rows.append({"Dataset": DS[ds], "Split": split, "Model": NAMES[model], "Comparison": f"{ref} vs {v}",
                         "Metric": "PR-AUC" if m == "pr_auc" else "F1", "n": len(s),
                         "Mean diff": d.mean(), "Median diff": np.median(d),
                         "W": st.statistic if st else np.nan, "Exact p": st.pvalue if st else 1.0})
    return pd.DataFrame(rows)


def with_holm(parts, ref="G0"):
    parts = [x for x in parts if not x.empty]
    if not parts:
        return pd.DataFrame()
    t = pd.concat(parts, ignore_index=True)
    ok = t["Exact p"].notna()
    t["Holm p"] = np.nan
    if ok.any():
        t.loc[ok, "Holm p"] = holm(t.loc[ok, "Exact p"].values)   # one family = all tested rows of the table
    t["Direction"] = np.where(~ok, "not tested (n < 6)", np.where(t["Holm p"] < 0.05,
                              np.where(t["Mean diff"] > 0, "HGCC graph better", "control better"),
                              "no significant difference"))
    return t


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs", required=True); ap.add_argument("--out", required=True)
    a = ap.parse_args(); os.makedirs(a.out, exist_ok=True)
    r = pd.concat([pd.read_csv(f) for f in glob.glob(os.path.join(a.runs, "*.csv"))], ignore_index=True)
    check_runs(r)
    w = pd.ExcelWriter(os.path.join(a.out, "revised_tables.xlsx"))
    tabs = {}
    gnn = ["sage", "gat", "grnn"]
    tabs["T16_with_HGCC"] = perf_table(r, [(d, m, "G0") for d in ("ieee", "ccf") for m in gnn],
                                       "GNNs on the HGCC graph (G0), temporal split, test set")
    tabs["T15_without_HGCC"] = perf_table(r, [(d, m, "G2") for d in ("ieee", "ccf") for m in gnn],
                                          "Same GNNs without the HGCC graph (G2: transaction features only)")
    tabs["T17_baselines"] = perf_table(r, [(d, m, v) for d in ("ieee", "ccf")
                                           for m, v in [("rf", "tabular"), ("xgb", "tabular"), ("knn", "tabular"),
                                                        ("sage", "G0"), ("gat", "G0"), ("grnn", "G0")]],
                                       "Tabular baselines vs HGCC-GNNs, temporal split, test set")
    # ablation (Table 7) and tests (Table 8)
    abl = []
    for ds in ("ccf", "ieee"):
        for split in ("temporal", "random"):
            for v, lab in [("G0", "HGCC graph"), ("G1", "Random edges (degree-preserving permutation)"),
                           ("G2", "No entity edges")]:
                x = r[(r.dataset == ds) & (r.split == split) & (r.model == "sage") & (r.variant == v)]
                if x.empty:
                    continue
                rec = {"Dataset": DS[ds], "Split": split, "Variant": lab, "Seeds": len(x)}
                rec.update({m: ms(x[m]) for m in ["pr_auc", "f1", "precision", "recall", "roc_auc"]})
                rec["Threshold (val-selected), median"] = f"{x.threshold.median():.3f}"
                abl.append(rec)
    tabs["T7_ablation"] = pd.DataFrame(abl)
    tests = [paired(r, ds, sp, "sage", "G0", ["G1", "G2"]) for ds in ("ccf", "ieee") for sp in ("temporal", "random")]
    tabs["T8_wilcoxon_ablation"] = with_holm(tests)
    tests = [paired(r, ds, "temporal", m, "G0", ["G2"]) for ds in ("ccf", "ieee") for m in gnn]
    tabs["T_HGCC_vs_without"] = with_holm(tests)
    # confusion matrices (seed 0, test split)
    cm = []
    for ds in ("ieee", "ccf"):
        for m in gnn:
            x = r[(r.dataset == ds) & (r.split == "temporal") & (r.model == m) & (r.variant == "G0") & (r.seed == 0)]
            if x.empty:
                continue
            x = x.iloc[0]
            cm.append({"Dataset": DS[ds], "Model": NAMES[m], "TP": x.tp, "FN": x.fn, "FP": x.fp, "TN": x.tn,
                       "Test transactions": x.n_test, "Test frauds": x.n_test_fraud,
                       "Precision = TP/(TP+FP)": round(x.precision, 4), "Recall = TP/(TP+FN)": round(x.recall, 4),
                       "F1": round(x.f1, 4), "Accuracy": round(x.accuracy, 4), "Threshold": round(x.threshold, 4)})
    tabs["T11_12_confusion_seed0"] = pd.DataFrame(cm)
    # timing (Table 13): mean over seeds, stages sum exactly to total by construction
    tm = []
    for ds in ("ieee", "ccf"):
        for m in gnn:
            for v in ("G0", "G2"):
                x = r[(r.dataset == ds) & (r.split == "temporal") & (r.model == m) & (r.variant == v)]
                if x.empty:
                    continue
                stages = x[["graph_build_seconds", "train_seconds", "inference_seconds", "other_seconds"]].mean()
                tot = x.run_seconds_total.mean()
                assert abs(stages.sum() - tot) < 1e-6
                tm.append({"Dataset": DS[ds], "Model": NAMES[m], "Graph": "HGCC (G0)" if v == "G0" else "none (G2)",
                           "Data loading (once)": round(x.load_seconds_once.mean(), 2),
                           "Graph construction": round(stages.graph_build_seconds, 2),
                           "Training (incl. validation checks)": round(stages.train_seconds, 2),
                           "Inference": round(stages.inference_seconds, 2),
                           "Other (metrics, I/O)": round(stages.other_seconds, 2),
                           "Per-run total": round(tot, 2), "Epochs (mean)": round(x.epochs_run.mean(), 1)})
    tabs["T13_timing"] = pd.DataFrame(tm)
    for k, t in tabs.items():
        t.to_csv(os.path.join(a.out, k + ".csv"), index=False)
        t.to_excel(w, sheet_name=k[:31], index=False)
        print(f"\n=== {k} ===\n{t.to_string(index=False)}")
    w.close()


if __name__ == "__main__":
    main()


Writing /kaggle/working/hgcc/hgcc_editorial_tables.py


In [5]:
%%writefile /kaggle/working/hgcc/make_figure2.py
#!/usr/bin/env python3
"""Figure 2: PR-AUC and F1 (mean +- SD over seeds) of the tabular baselines and the
HGCC-GNNs, from the per-run CSVs. usage: make_figure2.py <runs dir> <out.png>"""
import glob
import os
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

runs = pd.concat([pd.read_csv(f) for f in glob.glob(os.path.join(sys.argv[1], "*.csv"))], ignore_index=True)
runs = runs[(runs.split == "temporal") & runs.variant.isin(["tabular", "G0"])]
order = [("rf", "Random\nForest"), ("xgb", "XGBoost"), ("knn", "K-NN"),
         ("sage", "HGCC-\nGraphSAGE"), ("gat", "HGCC-\nGAT"), ("grnn", "HGCC-\nGRNN")]
series = [("pr_auc", "PR-AUC", "#2a78d6"), ("f1", "F1", "#eb6834")]
plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 9})
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.9), sharey=True)
for ax, (ds, title) in zip(axes, [("ieee", "IEEE-CIS"), ("ccf", "Credit Card")]):
    x = np.arange(len(order)); w = 0.36
    for k, (m, lab, col) in enumerate(series):
        mu = [runs[(runs.dataset == ds) & (runs.model == a)][m].mean() for a, _ in order]
        sd = [runs[(runs.dataset == ds) & (runs.model == a)][m].std(ddof=1) for a, _ in order]
        ax.bar(x + (k - 0.5) * (w + 0.02), mu, w, yerr=sd, color=col, label=lab, capsize=2,
               error_kw={"elinewidth": 0.8, "ecolor": "#52514e"}, edgecolor="white", linewidth=0.8)
    ax.set_xticks(x); ax.set_xticklabels([n for _, n in order], fontsize=7.5)
    ax.set_title(title, fontsize=10)
    ax.set_ylim(0, 1); ax.grid(axis="y", color="#e5e4e0", linewidth=0.6); ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.axvline(2.5, color="#b0afa8", linewidth=0.8, linestyle=":")
axes[0].set_ylabel("Test-set score")
axes[0].legend(frameon=False, loc="upper left", ncol=2)
fig.tight_layout()
fig.savefig(sys.argv[2], dpi=300, facecolor="white")
print("saved", sys.argv[2])


Writing /kaggle/working/hgcc/make_figure2.py


In [6]:
%%writefile /kaggle/working/hgcc/test_causality.py
"""Automatic check that the models are causal: changing the features of
transactions after position c (in arrival order) must leave every prediction
at positions <= c unchanged, for every model; and CausalGroups.mean/attend
must equal a brute-force computation."""
import sys, numpy as np, torch
sys.path.insert(0, sys.argv[1])
import hgcc_editorial_experiments as H

rng = np.random.default_rng(0)
n = 3000
D = {"X": rng.normal(size=(n, 7)).astype(np.float32), "y": (rng.random(n) < 0.05).astype(np.int64),
     "ent": {"customer": rng.integers(0, 60, n), "card": rng.integers(0, 150, n), "merchant": rng.integers(0, 20, n)},
     "amt": rng.gamma(2, 50, n)}
masks = H.make_split(D, "temporal", 0)
# brute-force check of the cumulative aggregators
g = H.CausalGroups(D["ent"]["customer"]); Hm = torch.randn(n, 5)
bf = torch.stack([Hm[[j for j in range(i + 1) if D["ent"]["customer"][j] == D["ent"]["customer"][i]]].mean(0) for i in range(n)])
assert torch.allclose(g.mean(Hm), bf, atol=1e-5), "mean wrong"
S, V = torch.randn(n, 2), torch.randn(n, 2, 3)
def att(i):
    J = [j for j in range(i + 1) if D["ent"]["customer"][j] == D["ent"]["customer"][i]]
    w = torch.softmax(30 * torch.tanh(S[J] / 30), 0); return (w.unsqueeze(-1) * V[J]).sum(0)
assert torch.allclose(g.attend(S, V), torch.stack([att(i) for i in range(n)]), atol=1e-5), "attend wrong"
print("aggregators match brute force")
for variant in ("G0", "G1", "G2"):
    G, _ = H.build_inputs(D, masks, variant, 0)
    for name in ("sage", "gat", "grnn"):
        torch.manual_seed(0); m = H.make_model(name, G).eval()
        with torch.no_grad():
            p1 = m(G)
            G2 = dict(G); c = 2000
            x = G["x_txn"].clone(); x[c + 1:] = torch.randn_like(x[c + 1:]) * 5; G2["x_txn"] = x
            p2 = m(G2)
        assert torch.equal(p1[:c + 1], p2[:c + 1]), f"{name}/{variant}: earlier predictions changed"
        assert not torch.equal(p1[c + 1:], p2[c + 1:]), f"{name}/{variant}: later predictions unaffected (test is vacuous)"
        print(f"{name:5s} {variant}: causal OK")
# gradient of the custom within-group cumulative sum equals autograd of a brute-force version
Hg = torch.randn(n, 3, requires_grad=True); W = torch.randn(n, 3)
(g.cumsum(Hg) * W).sum().backward(); g1 = Hg.grad.clone(); Hg.grad = None
ent = torch.from_numpy(D["ent"]["customer"]); same = (ent[:, None] == ent[None, :]) & (torch.arange(n)[None, :] <= torch.arange(n)[:, None])
((same.float() @ Hg) * W).sum().backward()
assert torch.allclose(g1, Hg.grad, atol=1e-4), "custom backward wrong"
print("custom backward matches brute force")


Writing /kaggle/working/hgcc/test_causality.py


In [7]:
%%writefile /kaggle/working/hgcc/run_kaggle.py
#!/usr/bin/env python3
"""
run_kaggle.py -- runs every experiment of the editorial revision and packs the
results into /kaggle/working/results.zip.

Steps
  0. find CODE PACKS (folders processed_ccf and processed_cis) under /kaggle/input;
     build the Credit Card graph with hgcc_build_ccf.py (fully specified HGCC)
  1. causality test of all models (CPU; aborts if any check fails)
  2. tabular baselines (CPU, in parallel with the GNN runs)
  3. learning-rate selection for each GNN on the Credit Card validation split
     (seed 0, HGCC graph, temporal split; lr in {0.001, 0.003, 0.01})
  4. all GNN runs (GPU if available), 10 seeds each
  5. tables, tests, consistency checks and Figure 2
  6. results.zip (per-run CSVs, logs, tables, predictions, checkpoints, code,
     environment)

Re-running the script skips runs that are already in the CSVs.
"""
import glob
import json
import os
import platform
import shutil
import subprocess
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))
OUT = os.environ.get("HGCC_OUT", "/kaggle/working/results")
SEEDS = os.environ.get("HGCC_SEEDS", "0 1 2 3 4 5 6 7 8 9").split()
LRS = os.environ.get("HGCC_LRS", "0.001 0.003 0.01").split()
EPOCHS = os.environ.get("HGCC_EPOCHS")            # only for smoke tests
BASE_MODELS = os.environ.get("HGCC_BASELINES", "rf xgb knn")
ZIP = os.path.join(os.path.dirname(OUT), "results")
T0 = time.time()


def log(msg):
    print(f"[{(time.time() - T0) / 60:7.1f} min] {msg}", flush=True)


def find_data():
    if os.environ.get("HGCC_DATA"):
        return os.environ["HGCC_DATA"]
    for root, dirs, _ in os.walk("/kaggle/input"):
        if "processed_ccf" in dirs and "processed_cis" in dirs:
            return root
    sys.exit("ERROR: no folder containing processed_ccf and processed_cis under /kaggle/input. "
             "Add the CODE PACKS dataset to this notebook (Add Input).")


def exp(args, log_name, env=None):
    ccf = ["--ccf-file", f"{OUT}/graph_ccf/transactions.csv"] if "ccf" in args else []
    cmd = [sys.executable, "-u", os.path.join(HERE, "hgcc_editorial_experiments.py"), "--data", DATA] + ccf + [
           "--threads", "2"] + (["--epochs", EPOCHS] if EPOCHS else []) + ["--pred-dir", f"{OUT}/predictions", "--ckpt-dir", f"{OUT}/checkpoints"] + args
    with open(f"{OUT}/logs/{log_name}.log", "a") as fh:
        r = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT, env=env)
    if r.returncode != 0:
        log(f"FAILED: {log_name} (see logs/{log_name}.log)")
        print(open(f"{OUT}/logs/{log_name}.log").read()[-1500:], flush=True)
        return False
    return True


def count(csv):
    return max(0, sum(1 for _ in open(csv)) - 1) if os.path.exists(csv) else 0


if __name__ == "__main__":
    DATA = find_data()
    for d in ("runs", "logs", "predictions", "checkpoints", "lrsel", "tables"):
        os.makedirs(f"{OUT}/{d}", exist_ok=True)
    import torch
    env = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.cuda.is_available(),
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           "cpu_count": os.cpu_count(), "platform": platform.platform(), "data": DATA}
    for m in ("numpy", "pandas", "sklearn", "scipy", "xgboost", "imblearn"):
        env[m] = __import__(m).__version__
    json.dump(env, open(f"{OUT}/environment.json", "w"), indent=1)
    log(f"data: {DATA}")
    log(f"environment: {env}")

    # 0b. Credit Card graph: fully specified HGCC latent-group inference (hgcc_build_ccf.py)
    CCF = f"{OUT}/graph_ccf"
    if not os.path.exists(f"{CCF}/graph_facts.json"):
        log("building the Credit Card HGCC graph (k selection, Edge-Aware K-means, causal assignment)")
        with open(f"{OUT}/logs/build_ccf.log", "w") as fh:
            r = subprocess.run([sys.executable, "-u", os.path.join(HERE, "hgcc_build_ccf.py"), "--txn",
                                os.path.join(DATA, "processed_ccf", "transactions.csv"), "--out", CCF], stdout=fh, stderr=subprocess.STDOUT)
        if r.returncode != 0:
            print(open(f"{OUT}/logs/build_ccf.log").read()[-2000:]); sys.exit("graph build FAILED")
    log("Credit Card graph: " + json.dumps(json.load(open(f"{CCF}/graph_facts.json"))["chosen_k"]))

    # 1. causality test
    r = subprocess.run([sys.executable, os.path.join(HERE, "test_causality.py"), HERE], capture_output=True, text=True)
    open(f"{OUT}/causality_test.txt", "w").write(r.stdout + r.stderr)
    if r.returncode != 0:
        print(r.stdout, r.stderr); sys.exit("causality test FAILED")
    log("causality test passed:\n" + r.stdout)

    # 2. baselines on CPU, in the background (5 seeds; the tree/KNN models are nearly deterministic)
    BASE_SEEDS = os.environ.get("HGCC_BASE_SEEDS", "0 1 2 3 4")
    def base_line(ds):
        ccf = f'--ccf-file "{OUT}/graph_ccf/transactions.csv" ' if ds == "ccf" else ""
        return (f'"{sys.executable}" -u "{HERE}/hgcc_editorial_experiments.py" --data "{DATA}" --dataset {ds} {ccf}'
                f'--models {BASE_MODELS} --seeds {BASE_SEEDS} --out "{OUT}/runs/{ds}_temporal_baselines.csv" '
                f'--pred-dir "{OUT}/predictions" >> "{OUT}/logs/{ds}_baselines.log" 2>&1')
    baselines = subprocess.Popen(["bash", "-c", " && ".join(base_line(ds) for ds in ("ccf", "ieee"))])
    log("baselines started in the background (CPU)")

    # 3. learning rates: fixed from an earlier selection run (HGCC_LR_FIXED) or selected here
    import pandas as pd
    import threading
    fixed = os.environ.get("HGCC_LR_FIXED")
    if fixed:
        chosen = {m: str(v) for m, v in json.loads(fixed).items()}
        log(f"learning rates fixed from the earlier selection run: {chosen}")
    else:
        chosen = {}
        for m in ("sage", "gat", "grnn"):
            res = []
            for lr in LRS:
                csv = f"{OUT}/lrsel/ccf_{m}_lr{lr}.csv"
                if count(csv) == 0:
                    exp(["--dataset", "ccf", "--split", "temporal", "--models", m, "--variants", "G0", "--seeds", "0",
                         "--lr", lr, "--out", csv], f"lrsel_{m}")
                x = pd.read_csv(csv).iloc[0]
                res.append((float(x.best_val_pr_auc), lr))
                log(f"lr selection {m} lr={lr}: validation PR-AUC {x.best_val_pr_auc:.4f} ({x.epochs_run:.0f} epochs)")
            chosen[m] = max(res)[1]
            log(f"==> {m}: lr = {chosen[m]}")
    pd.DataFrame([{"model": m, "chosen_lr": float(v)} for m, v in chosen.items()]).to_csv(f"{OUT}/lr_selection.csv", index=False)

    # 4. GNN runs: one worker per GPU; seeds in the outer loop so that a run stopped by the time
    #    budget still has the same seeds for every comparison
    BUDGET = float(os.environ.get("HGCC_BUDGET_H", "10.5")) * 3600
    N_SAGE = int(os.environ.get("HGCC_SEEDS_ABLATION", "10")); N_OTHER = int(os.environ.get("HGCC_SEEDS_OTHER", "5"))
    seeds = [int(x) for x in SEEDS]
    def plan(ds):
        jobs = []
        for s in seeds[:N_SAGE]:
            jobs.append((ds, "temporal", "sage", ["G0", "G1", "G2"], s))
            if s in seeds[:N_OTHER]:
                if ds == "ccf":
                    jobs.append((ds, "random", "sage", ["G0", "G1"], s))
                jobs.append((ds, "temporal", "gat", ["G0", "G2"], s))
                jobs.append((ds, "temporal", "grnn", ["G0", "G2"], s))
        return jobs
    ngpu = torch.cuda.device_count()
    queues = [plan("ccf"), plan("ieee")] if ngpu >= 2 else [plan("ccf") + plan("ieee")]
    def worker(q, gpu):
        envw = dict(os.environ)
        if gpu is not None:
            envw["CUDA_VISIBLE_DEVICES"] = str(gpu)
        for ds, split, m, variants, s in q:
            if time.time() - T0 > BUDGET:
                log(f"time budget reached: not starting {ds}/{split}/{m} seed {s}")
                return
            name = f"{ds}_{split}_{m}"
            t1 = time.time()
            ok = exp(["--dataset", ds, "--split", split, "--models", m, "--variants"] + variants +
                     ["--seeds", str(s), "--lr", chosen[m], "--out", f"{OUT}/runs/{name}.csv"], name, envw)
            log(f"{'done' if ok else 'FAILED'} {name} seed {s} ({(time.time() - t1) / 60:.1f} min, GPU {gpu})")
    threads = [threading.Thread(target=worker, args=(q, i if ngpu else None)) for i, q in enumerate(queues)]
    log(f"{len(threads)} GPU worker(s); jobs: " + ", ".join(str(len(q)) for q in queues))
    for th in threads:
        th.start()
    for th in threads:
        th.join()

    # wait for the baselines (stop them if the time budget is exhausted)
    while baselines.poll() is None:
        if time.time() - T0 > BUDGET + 1800:
            baselines.kill(); log("baselines stopped by the time budget"); break
        log(f"waiting for baselines: ccf {count(f'{OUT}/runs/ccf_temporal_baselines.csv')}, "
            f"ieee {count(f'{OUT}/runs/ieee_temporal_baselines.csv')}")
        time.sleep(120)
    log("baselines finished")

    # 5. tables and figure
    r = subprocess.run([sys.executable, os.path.join(HERE, "hgcc_editorial_tables.py"), "--runs", f"{OUT}/runs",
                        "--out", f"{OUT}/tables"], capture_output=True, text=True)
    open(f"{OUT}/tables/tables_stdout.txt", "w").write(r.stdout + r.stderr)
    if r.returncode != 0:
        log("tables FAILED (runs are still packed):\n" + r.stdout[-2000:] + r.stderr[-2000:])
    else:
        print(r.stdout, flush=True)
    subprocess.run([sys.executable, os.path.join(HERE, "make_figure2.py"), f"{OUT}/runs", f"{OUT}/tables/figure2.png"])

    # 6. pack
    os.makedirs(f"{OUT}/code", exist_ok=True)
    for f in glob.glob(os.path.join(HERE, "*.py")) + glob.glob(os.path.join(HERE, "*.md")):
        shutil.copy(f, f"{OUT}/code/")
    shutil.make_archive(ZIP, "zip", OUT)
    log(f"ALL DONE -> {ZIP}.zip ({os.path.getsize(ZIP + '.zip') / 1e6:.1f} MB)")


Writing /kaggle/working/hgcc/run_kaggle.py


In [8]:
# learning rates selected on validation in the first run (GraphSAGE 0.01, GAT 0.003, GRNN 0.01)
!HGCC_LR_FIXED='{"sage": 0.01, "gat": 0.003, "grnn": 0.01}' python -u /kaggle/working/hgcc/run_kaggle.py

[    0.1 min] data: /kaggle/input/datasets/egwusam/tosin-new-dataset/CODE PACKS
[    0.1 min] environment: {'python': '3.13.16', 'torch': '2.11.0+cu130', 'cuda': True, 'gpu': 'Tesla T4', 'cpu_count': 4, 'platform': 'Linux-6.18.55+-x86_64-with-glibc2.39', 'data': '/kaggle/input/datasets/egwusam/tosin-new-dataset/CODE PACKS', 'numpy': '2.1.3', 'pandas': '2.3.3', 'sklearn': '1.6.1', 'scipy': '1.16.3', 'xgboost': '3.4.1', 'imblearn': '0.14.2'}
[    0.1 min] building the Credit Card HGCC graph (k selection, Edge-Aware K-means, causal assignment)
[    5.7 min] Credit Card graph: {"customer": 1000, "merchant": 1000}
[    5.8 min] causality test passed:
aggregators match brute force
sage  G0: causal OK
gat   G0: causal OK
grnn  G0: causal OK
sage  G1: causal OK
gat   G1: causal OK
grnn  G1: causal OK
sage  G2: causal OK
gat   G2: causal OK
grnn  G2: causal OK
custom backward matches brute force

[    5.8 min] baselines started in the background (CPU)
[    5.8 min] learning rates fixed from the

In [9]:
from IPython.display import FileLink
FileLink('results.zip')

/kaggle/working/results.zip